# 7.04 读懂工业级代码库：openpi、FluxVLA 与实时分块（RTC）

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/07_vla/7.04_codebase_reading.ipynb)


## 为什么要学

自己写的 mini-π0 只有几百行。openpi、FluxVLA、RLinf 动辄几万行，但核心模型代码你已经全部写过了。剩下的大部分代码在解决**工程问题**：数据格式转换、归一化、多机器人适配、分布式训练、部署通信、实时性。

本课给出三个代码库的"导航地图"，用 0.2 课的方法（**追一条数据**）把关键路径串起来，最后**动手实现 RTC 实时分块**（替换式与引导式两种修补方法）：这是把 VLA 部署到真机上时最常遇到的问题之一。

---

## 1. openpi：一条数据从机器人到动作的完整旅程

### 1.1 目录地图（只列关键文件）

```text
openpi/
├── src/openpi/
│   ├── models/               # JAX 版模型（官方主线）：pi0.py、pi0_fast.py、gemma.py、siglip.py、tokenizer.py
│   ├── models_pytorch/       # PyTorch 版：pi0_pytorch.py（7.02 课精读过）、gemma_pytorch.py
│   ├── policies/             # 机器人适配：aloha_policy.py、droid_policy.py、libero_policy.py
│   ├── training/config.py    # 所有训练配置（TrainConfig + DataConfig），例如 pi05_aloha、pi0_aloha_towel
│   ├── transforms.py         # 数据变换流水线：Repack、Normalize、ResizeImages、TokenizePrompt、PadStatesAndActions……
│   └── serving/              # websocket 推理服务端
├── scripts/
│   ├── compute_norm_stats.py # 计算归一化统计量（1.01 课的 q01/q99）
│   ├── train.py / train_pytorch.py
│   └── serve_policy.py       # 启动推理服务
└── packages/openpi-client/   # 机器人端客户端：websocket_client_policy.py、action_chunk_broker.py
```

### 1.2 训练时：数据变换流水线

`DataConfig` 把变换分成三组，**顺序执行**：

```text
LeRobot 数据集的一帧
  │ repack_transforms   改名：把数据集里的 key 映射成 openpi 的标准 key（observation/images/... -> images/cam_high）
  │ data_transforms     机器人相关：AlohaInputs（关节翻转、夹爪角度换算）、DeltaActions（可选：动作改为相对当前状态的增量）
  │ Normalize           用 norm_stats 归一化 state / actions（π0.5 用分位数 q01/q99）
  │ model_transforms    模型相关：ResizeImages(224)、TokenizePrompt、PadStatesAndActions(32 维)
  ▼
模型输入（Observation + actions）
```

以 ALOHA 为例（`policies/aloha_policy.py`）：

- **`_joint_flip_mask()`** = `[1, -1, -1, 1, 1, 1, 1, ...]`：ALOHA 的某些关节方向与 π 的内部约定相反，需要翻转；
- **夹爪换算**：ALOHA 夹爪的线性位置与 π 内部使用的角度表示不同，`_gripper_to_angular` 做转换；
- **`adapt_to_pi=True`**：如果你用的是 `pi0_base` / `pi05_base` 预训练权重，必须打开，让你的数据与预训练时的约定一致。

> **真机最常见的 bug 就在这一层**：关节方向、夹爪开合方向、单位（度/弧度）、相机名称与顺序和预训练约定不一致。训练 loss 看起来正常，部署时动作却完全不对。排查方法：把归一化后的 state/action 统计量打印出来，和预训练数据集的分布对比；把一条示教的动作原样回放到机器人上。

### 1.3 推理时：服务端 + 客户端

```text
机器人端（openpi-client）                          GPU 服务器（scripts/serve_policy.py）
  obs = {images, state, prompt}
  ActionChunkBroker.infer(obs)
     └─ 当前动作块用完了？──是──> WebsocketClientPolicy ──msgpack──> Policy.infer
                                                                     ├─ 输入变换（同训练）
                                                                     ├─ model.sample_actions（KV cache + 10 步去噪）
                                                                     ├─ Unnormalize（反归一化）
                                                                     └─ 输出变换（AlohaOutputs：翻转回去、夹爪换算回去）
       <─────────────────────────── 动作块 (50, 14) ──────────────────┘
     └─ 否：返回块中的下一个动作
```

`ActionChunkBroker`：拿到一个动作块后逐个返回，**用完了才发起下一次推理**。这是最简单的"同步分块执行"：推理期间机器人要么停下等待，要么只能继续执行旧块，这就引出了第 3 节的 RTC。

---

## 2. FluxVLA：配置驱动的模块化 VLA 平台

FluxVLA（[GitHub](https://github.com/FluxVLA/FluxVLA)）的设计理念是"统一配置、标准接口、模块解耦"。理解它只需要两件事。

### 2.1 注册表 + 配置字典

所有组件都通过注册表创建：

```py
@VLAS.register_module()
class PI05FlowMatching(PI0FlowMatching):
    ...

# configs/pi05/pi05_paligemma_aloha_full_finetune.py
model = dict(
    type='PI05FlowMatching',                          # 从注册表里找到这个类
    llm_backbone=dict(type='ConditionGemmaModel', hidden_size=2048, num_hidden_layers=18, num_attention_heads=8,
                      num_key_value_heads=1, head_dim=256, ...),
    vision_backbone=dict(type='SigLIPViTBackbone', vision_config=dict(hidden_size=1152, num_hidden_layers=27, patch_size=14, ...)),
    projector=dict(type='LinearProjector', in_dim=1152, out_dim=2048),
    llm_expert=dict(type='ConditionGemmaModel', hidden_size=1024, use_adarms=True, adarms_cond_dim=1024, ...),
    action_in_proj=dict(type='LinearProjector', in_dim=32, out_dim=1024),
    time_mlp_in=..., time_mlp_out=..., n_action_steps=50, max_action_dim=32, ori_action_dim=14,
    pretrained_name_or_path='./checkpoints/pi05_base/model.safetensors',
    name_mapping={'llm_expert': 'paligemma_with_expert.gemma_expert.model', ...},   # 把 openpi 权重名映射到自己的模块
)
```

对照你学过的内容逐项检查：`llm_backbone` = 7.01 的 Gemma-2B；`llm_expert` = 7.02 的动作专家（宽 1024、`use_adarms=True` 即 π0.5 的 adaRMSNorm）；`projector` = SigLIP 到 Gemma 的线性投影；`max_action_dim=32` 是 openpi 的 32 维动作补零约定，ALOHA 实际 14 维。**你现在能读懂这个配置里的每一个字段。**

### 2.2 目录地图

```text
fluxvla/
├── models/
│   ├── vlas/          # 完整的 VLA：pi0_flowmatching.py、pi05_flowmatching.py、pi05_flowmatching_inference_rtc.py、smolvla、gr00t……
│   ├── backbones/     # 视觉 / 语言 / VLM 骨干
│   ├── heads/         # 动作头：flow_matching_head.py（GR00T 风格的 DiT 头，含 RTC）、openvla_head.py……
│   └── projectors/
├── datasets/          # ParquetDataset（LeRobot v2 格式）等
├── engines/runners/   # 训练（ddp/fsdp_train_runner）、真机推理（aloha_inference_runner、*_rtc_inference_runner）
└── ...
```

同一个"组件"可以在不同 VLA 之间复用。比如把视觉骨干从 SigLIP 换成 Qwen3-VL，只需要改配置。这比 openpi 的"每个模型一个大类"灵活，代价是要多跳几层才能找到真正的代码。读代码的技巧：**从 config 的 `type` 出发，搜索 `register_module` 找到类，再看它的 `forward` / `predict_action`**。

---

## 3. 实时分块（RTC）：动手实现

### 3.1 问题

假设推理一次需要 $d$ 个控制周期（π0 在 4090 上约 70~100ms，50Hz 控制下 $d\approx4\sim5$）。有两种朴素做法：

1. **停下来等**：动作块执行完，机器人停住等推理。动作一顿一顿的，还会让速度相关的任务失败；
2. **边执行边推理**：在执行旧块的同时用当前观测推理新块，推理完成时已经过去了 $d$ 步，**新块的前 $d$ 个动作已经"过时"**，直接切换到新块的第 $d$ 个动作。问题：新旧两个块是独立采样的，可能选择了不同的模式或有不同的偏差，**切换点会出现跳变**。

### 3.2 RTC 的思路（Black, Galliker, Levine 2025）

在推理新块时，**已经知道**接下来 $d$ 步机器人会执行旧块的动作 $\mathbf a^{prev}_{1:d}$。所以把"新块的前 $d$ 步 = 旧块的这 $d$ 步"作为**约束**，让生成模型去"补全"后面的动作。这就是图像生成中的 **inpainting（修补）**。两种做法：

**(1) 替换式修补**（最简单）：去噪的每一步，把前 $d$ 个位置强制替换成"按当前时间加噪后的旧动作"：$\mathbf x_t[1{:}d]\leftarrow t\boldsymbol\epsilon[1{:}d]+(1-t)\mathbf a^{prev}_{1:d}$。FluxVLA 中 `_predict_action_prefix_rtc` 是这个思路（并配合了逐位置的时间编码）。

**(2) 引导式修补**（RTC 论文的方法；FluxVLA 的 `rtc_guidance.py` 移植自 PI 的 [real-time-chunking-kinetix](https://github.com/Physical-Intelligence/real-time-chunking-kinetix)）：每一步先估计"去噪完成后的动作" $\hat{\mathbf x}_0$，再计算它与旧动作的误差，用这个误差**修正速度**：

$$
\hat{\mathbf x}_0=\mathbf x_t-t\,\mathbf v_\theta(\mathbf x_t,t)\quad\text{（openpi 约定下由 } \mathbf x_t=t\boldsymbol\epsilon+(1-t)\mathbf x_0,\ \mathbf v=\boldsymbol\epsilon-\mathbf x_0 \text{ 推出）}
$$

$$
\mathbf v\leftarrow\mathbf v-g(\tau)\cdot W\odot\big(\mathbf a^{prev}-\hat{\mathbf x}_0\big),\qquad\tau=1-t
$$

- $W$：逐位置权重。前 $d$ 个位置为 1，之后指数衰减到 0（软约束：让衔接处平滑过渡）；
- $g(\tau)=\min\Big(\frac{1-\tau}{\tau}\cdot\frac{\tau^2+(1-\tau)^2}{(1-\tau)^2},\ g_{max}\Big)$：去噪早期引导强、后期弱（与 FluxVLA 的 `compute_guidance_weight` 相同）；
- 符号：FluxVLA 的 GR00T 头用"$t=0$ 噪声、$t=1$ 干净"的约定，速度方向与 openpi 相反，所以那里是 `v + gw * error`，openpi 约定下是减号。**跨代码库移植时，时间方向的约定是最容易出错的地方。**

### 3.3 实验设置：一个"两种做法都对"的场景

RTC 最重要的作用之一，是在**两种模式都合理**时保持新旧动作块一致（否则机器人会犹豫、来回摆动）。我们回到第 6 章的双峰障碍物任务，训练一个流匹配策略，然后在**起点**处测试：给定一个"向左绕"的旧动作块，新动作块还会继续向左吗？

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt
from vla_ladder.envs import ObstacleReachEnv, collect_demos
from vla_ladder.models.flow import ChunkDenoiser, flow_matching_loss
from vla_ladder.utils import get_device, set_seed

device = get_device()
H = 16
env = ObstacleReachEnv(num_envs=100, seed=0)
demos = collect_demos(env, 200, seed=0)
Os, As = [], []
for d_ in demos:
    T_ = len(d_["actions"]); idx = np.minimum(np.arange(T_)[:, None] + np.arange(H), T_ - 1)
    Os.append(d_["obs"]); As.append(d_["actions"][idx])
O = torch.tensor(np.concatenate(Os)).to(device); A = torch.tensor(np.concatenate(As)).to(device)
set_seed(0)
policy = ChunkDenoiser(2, 2, H).to(device)
opt = torch.optim.AdamW(policy.parameters(), 1e-3); sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, 4000)
for step in range(4000):
    i = torch.randint(0, len(O), (256,), device=device)
    loss = flow_matching_loss(policy, A[i], O[i]); opt.zero_grad(); loss.backward(); opt.step(); sched.step()
policy.eval()

@torch.no_grad()
def sample_plain(obs, steps=10):
    B = obs.shape[0]; x = torch.randn(B, H, 2, device=device); dt, t = -1.0 / steps, 1.0
    while t >= -dt / 2:
        x = x + dt * policy(x, torch.full((B,), t, device=device), obs); t += dt
    return x.clamp(-1, 1)

@torch.no_grad()
def sample_replace(obs, prev, d, steps=10):
    """替换式修补"""
    B = obs.shape[0]; noise = torch.randn(B, H, 2, device=device); x = noise.clone(); dt, t = -1.0 / steps, 1.0
    while t >= -dt / 2:
        x[:, :d] = t * noise[:, :d] + (1 - t) * prev[:, :d]
        x = x + dt * policy(x, torch.full((B,), t, device=device), obs); t += dt
    x[:, :d] = prev[:, :d]
    return x.clamp(-1, 1)

o0 = torch.tensor([[0.0, -0.8]], device=device).repeat(1000, 1)       # 起点：左右两种绕法都可以
base = sample_plain(o0)
left_chunk = base[base[:, :, 0].sum(1) < 0][:1].repeat(1000, 1, 1)    # 取一个"向左"的动作块作为旧块
goes_left = lambda x, d: (x[:, d:, 0].sum(1) < 0).float().mean().item()
print("独立采样（不做 RTC）：新块向左的比例", round(goes_left(sample_plain(o0), 0), 3))
for d in [1, 3, 5]:
    print(f"替换式修补 d={d}：新块（前缀之后）向左的比例 {goes_left(sample_replace(o0, left_chunk, d), d):.3f}")

替换式修补的约束力很弱：即使前 3 步已经确定"向左"，后面仍有相当比例的样本转去右边，这正是 RTC 论文要解决的问题。

### 3.4 练习：实现引导式 RTC

In [ ]:
def prefix_weights(H, d, decay_end=None):
    """前 d 个位置为 1，之后按 exp(-3·frac) 衰减到 decay_end（默认 2d）为止（与 FluxVLA compute_prefix_weights 的 'exp' 相同）"""
    decay_end = decay_end or 2 * d
    w = torch.zeros(H, device=device); w[:d] = 1.0
    L = decay_end - d
    for i in range(max(L, 0)):
        if d + i >= H:
            break
        w[d + i] = math.exp(-3.0 * i / max(L - 1, 1))
    return w

def guidance_weight(tau, max_g=5.0):
    """tau = 1 - t（去噪进度，0 = 纯噪声，1 = 干净）"""
    tau, omt = max(tau, 1e-6), max(1 - tau, 1e-6)
    return min((omt / tau) * (tau ** 2 + omt ** 2) / omt ** 2, max_g)

@torch.no_grad()
def sample_guided(obs, prev, d, steps=10, max_g=5.0):
    B = obs.shape[0]; x = torch.randn(B, H, 2, device=device); dt, t = -1.0 / steps, 1.0
    W = prefix_weights(H, d)[None, :, None]
    while t >= -dt / 2:
        # TODO:
        # 1) v = policy(x, 全为 t 的 (B,), obs)
        # 2) x0_hat = x - t * v                                  （openpi 约定下的"去噪完成后的动作"估计）
        # 3) v = v - guidance_weight(1 - t, max_g) * W * (prev - x0_hat)
        # 4) x = x + dt * v；t += dt
        raise NotImplementedError
    return x.clamp(-1, 1)

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def prefix_weights(H, d, decay_end=None):
    """前 d 个位置为 1，之后按 exp(-3·frac) 衰减到 decay_end（默认 2d）为止（与 FluxVLA compute_prefix_weights 的 'exp' 相同）"""
    decay_end = decay_end or 2 * d
    w = torch.zeros(H, device=device); w[:d] = 1.0
    L = decay_end - d
    for i in range(max(L, 0)):
        if d + i >= H:
            break
        w[d + i] = math.exp(-3.0 * i / max(L - 1, 1))
    return w

def guidance_weight(tau, max_g=5.0):
    """tau = 1 - t（去噪进度，0 = 纯噪声，1 = 干净）"""
    tau, omt = max(tau, 1e-6), max(1 - tau, 1e-6)
    return min((omt / tau) * (tau ** 2 + omt ** 2) / omt ** 2, max_g)

@torch.no_grad()
def sample_guided(obs, prev, d, steps=10, max_g=5.0):
    B = obs.shape[0]; x = torch.randn(B, H, 2, device=device); dt, t = -1.0 / steps, 1.0
    W = prefix_weights(H, d)[None, :, None]
    while t >= -dt / 2:
        v = policy(x, torch.full((B,), t, device=device), obs)
        x0_hat = x - t * v
        v = v - guidance_weight(1 - t, max_g) * W * (prev - x0_hat)
        x = x + dt * v
        t += dt
    return x.clamp(-1, 1)

In [ ]:
# ✅ 自测
for d in [1, 3, 5]:
    xg = sample_guided(o0, left_chunk, d)
    print(f"引导式 RTC d={d}：新块向左的比例 {goes_left(xg, d):.3f}，前缀与旧动作的平均误差 {(xg[:, :d] - left_chunk[:, :d]).abs().mean():.4f}")
assert goes_left(sample_guided(o0, left_chunk, 3), 3) > 0.9
print("通过 ✅ 引导式 RTC 让新块'延续'了旧块选择的模式")

paths = lambda x: np.cumsum(x.cpu().numpy() * 0.05, 1) + np.array([0.0, -0.8])
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (name, x) in zip(axes, [("independent", sample_plain(o0)), ("replace d=3", sample_replace(o0, left_chunk, 3)), ("guided d=3", sample_guided(o0, left_chunk, 3))]):
    env.plot(title=name, ax=ax)
    for p in paths(x)[:150]:
        ax.plot(p[:, 0], p[:, 1], "b-", alpha=0.08)
    ax.plot(*paths(left_chunk[:1])[0].T, "r-", lw=2, label="previous chunk")
axes[0].legend(fontsize=7); plt.tight_layout(); plt.show()

> 在真机上，"新旧块不一致"会表现为**抖动、冲击、夹爪误开合，或在两种做法之间犹豫**。精细插线对此尤其敏感。你在 Piper 上用的 RTC bridge 解决的正是这个问题。另外要注意：如果策略在当前状态下本来就是单峰的（已经"选定了做法"），切换点的跳变主要来自**新观测带来的新信息**，RTC 不会、也不应该消除这种修正。配套代码包的 `vla_ladder.models.mini_pi0.sample_actions_rtc` 为 mini-π0 同时提供了两种方法（`method="replace"` / `"guidance"`）。

---

## 4. RLinf：VLA 强化学习框架（第 8 章预告）

RLinf 把 RL 训练拆成几种 **Worker**，由调度器按"放置策略"（placement）分配到 GPU 上：

| Worker | 作用 | 对应你学过的 |
|---|---|---|
| env worker | 跑仿真/真机环境，产生观测、执行动作 | 2.x 课的环境循环 |
| rollout worker | 用当前策略推理动作（VLA 前向） | `sample_actions` |
| actor worker | 计算损失、更新参数（FSDP） | PPO（2.11）/ SAC（2.12）/ RLT 的 actor-critic |

RLT 相关代码：`rlinf/models/embodiment/modules/rlt_token_transformer.py`（RL token 编码器/解码器）、`rlinf/models/embodiment/mlp_policy/rlt_mlp_policy.py`（actor-critic 头）、`rlinf/workers/actor/fsdp_rlt_ac_policy_worker.py`（训练逻辑）。下一章从零实现它们的精简版。

---

## 小结

- openpi 的数据流：repack → 机器人适配（翻转/夹爪/增量动作）→ 归一化 → 模型变换（resize/分词/补齐到 32 维）→ 模型 → 反归一化 → 输出变换。真机 bug 多发生在机器人适配和归一化这两层。
- FluxVLA：注册表 + 配置字典；从 `type` 找类，从类找 `forward`。
- RTC：推理期间机器人在执行旧动作，用"修补"把新块的前 $d$ 步锚定在旧动作上。替换式修补约束力弱；引导式修补（RTC 论文 / FluxVLA）能可靠地保持新旧块的一致性。

🎉 **第 7 章完成！** 你已经具备了"自己写一个 VLA"的能力。最后一章：让它通过强化学习变得更好。